In [ ]:
import os
import json
from llms.coforgeAIGardenLLM import CoforgeAIGardenLLM
from llms.coforgeTextEmbeddings import CustomEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:


loader = Docx2txtLoader("Policy_RCA_Guide_DB_File_API.docx")
documents = loader.load()

print(documents[0].page_content[:500])

In [ ]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Total Chunks Created: {len(chunks)}")

#  Create embeddings
embeddings = CustomEmbeddings()

# Store in FAISS vector DB
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save FAISS index locally
vectorstore.save_local("rca_faiss_index")

print("FAISS index created and saved successfully ✅")


In [7]:
from langchain_community.vectorstores import FAISS

# Load the FAISS index
vectorstore = FAISS.load_local(
    "rca_faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

# Simple test query
query = "How to fix PG-FS-003 error?"

docs = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)

❓ Embedding query...

➡️  Calling embedding API
    URL        : https://quasarmarket.coforge.com/qag/llmrouter-api/v2/text/embeddings
    Model      : coforge-text-embedding
    Text chars : 27
✅ Received embedding | time=1.89s | dim=736


--- Result 1 ---
RCA – XML File Not Written (PG-FS-003)

Verification:
1. Policy exists in DB
2. XML missing from C:\policies\generated\

Fix (Windows):
mkdir C:\policies\generated
icacls C:\policies\generated /grant Everyone:F

Recovery SQL:
UPDATE policies SET policy_status='XML_RETRY', error_code=NULL, error_message=NULL WHERE error_code='PG-FS-003';

RCA – Invalid XML Content (PG-XML-002)

Verification:
XML file exists but contains invalid or missing data fields.

Fix rule: Correct underlying DB data, not XML.

SQL Fix Example:
UPDATE policies SET premium_amount='5000', policy_status='XML_RETRY', error_code=NULL WHERE policy_name='<POLICY_NAME>';

RCA – Database Failure (PG-DB-001)

Verification:
Check DB availability using:
SELECT 1;

--- Resul